# Self-Healing SOC — ML Model Training

Train an Isolation Forest anomaly detector on cybersecurity data (CIC-IDS2017 / UNSW-NB15 / custom CSV), then export `soc_model.joblib` and copy it into `backend/app/ml/` so the FastAPI backend uses it instead of the built-in baseline.

**Features used by the SOC backend:** `failed_logins`, `unique_ports`, `process_spawns`, `network_rate`

In [ ]:
%pip install -q scikit-learn pandas numpy joblib matplotlib

In [ ]:
from google.colab import files
uploaded = files.upload()
import io, os
import pandas as pd

frames = [pd.read_csv(io.BytesIO(v)) for v in uploaded.values()]
df = pd.concat(frames, ignore_index=True)
print('Loaded', df.shape)
df.head()

## Normalize columns to the 4 SOC features

In [ ]:
COLUMN_ALIASES = {
    'failed_logins': ['login_attempts', 'failed_login_attempts', 'failed_logins'],
    'unique_ports': ['dst_port', 'destination_port', 'dst_ports', 'service_port', 'unique_ports'],
    'process_spawns': ['process_count', 'proc_count', 'processes', 'process_spawns'],
    'network_rate': ['flow_bytes_s', 'bytes_per_second', 'network_rate', 'flow_duration_bytes'],
}

rename = {}
for target, aliases in COLUMN_ALIASES.items():
    if target not in df.columns:
        for alias in aliases:
            if alias in df.columns:
                rename[alias] = target
                break
df = df.rename(columns=rename)

FEATURES = ['failed_logins', 'unique_ports', 'process_spawns', 'network_rate']
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    raise ValueError(f'Missing feature columns after normalization: {missing}. Available: {list(df.columns)}')

X = df[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)
label_col = next((c for c in ['label','class','attack_cat','attack_type'] if c in df.columns), None)
y = df[label_col] if label_col else None
print('Training matrix:', X.shape, '| labels:', label_col)

## Train Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

model = IsolationForest(n_estimators=200, contamination=0.10, random_state=42)
model.fit(X)

pred = model.predict(X)          # -1 anomaly, 1 normal
scores = model.decision_function(X)
risk = np.clip(50 - 120 * scores, 0, 100)
anomaly_rate = (pred == -1).mean() * 100
print(f'anomaly rate: {anomaly_rate:.2f}%')
print(f'avg risk:     {risk.mean():.1f} / 100')

if y is not None:
    from sklearn.metrics import classification_report
    is_attack = ~y.astype(str).str.lower().isin(['normal','benign','0','0.0'])
    print(classification_report(is_attack.astype(int), (pred == -1).astype(int), digits=3))

## Export `soc_model.joblib`

In [ ]:
import joblib
joblib.dump({'model': model, 'features': FEATURES}, 'soc_model.joblib')
print('Exported soc_model.joblib — download it below, then place it at:')
print('  backend/app/ml/soc_model.joblib')

In [ ]:
from google.colab import files
files.download('soc_model.joblib')